# 16. Is the Temperature-Scaling Null Result a Sample-Size Artifact?

`DESIGN.md` §15.2/§24.3 reports that post-hoc temperature scaling gives **no measurable
calibration benefit** on this project's data (ECE 0.175 → 0.179 at small scale, 0.1637 →
0.1700 at full scale) and attributes this to `threshold_cal` being small (300 Imagenette
images) combined with the backbone already being close to well-calibrated - not a bug
(ranking preservation was checked and holds exactly).

That explanation was never directly tested. This notebook tests it the methodologically
correct way: **bootstrap-resampling the existing, legitimate `threshold_cal` split** at
several sizes and checking whether post-scaling ECE improves as the effective fitting
sample size grows. This stays entirely inside `combiner_fit`/`threshold_cal` data - `id_test`
is used only for measuring ECE afterward, exactly as in notebook 07, and is never
resampled or touched for fitting. (An earlier, discarded idea for this diagnostic - re-fitting
temperature scaling directly on the full 50,000-image ImageNet-1k set - would have violated
`DESIGN.md` §10.5's protocol, since that set is designated reporting-only, same rule as
`imagenet_a`/`imagenet_o`. This notebook does not do that.)

If ECE-after-scaling improves substantially as the resampled calibration size grows, the
null result is a sample-size artifact. If it stays flat even at 4× the real split's size,
that supports `DESIGN.md`'s existing "already well-calibrated" explanation instead.

In [1]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)

from deployment_reliability.calibration import TemperatureScaling
from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.features import featurize

## Load the cache and reproduce notebook 07's exact combiner fit

In [2]:
CACHE_PATH = os.path.join("..", "data", "logit_cache_resnet50.pt")
assert os.path.exists(CACHE_PATH), f"{CACHE_PATH} not found - run scripts/collect_logits.py resnet50 first."
cache = torch.load(CACHE_PATH)
logits, labels = cache["logits"], cache["labels"]
splits_arr = np.array(cache["splits"])

def mask(name):
    return torch.from_numpy(splits_arr == name)

m_fit, m_cal, m_test = [mask(n) for n in ("combiner_fit", "threshold_cal", "id_test")]
print(f"combiner_fit n={int(m_fit.sum())}  threshold_cal n={int(m_cal.sum())}  id_test n={int(m_test.sum())}")

predicted = logits.argmax(dim=-1)
correct = predicted == labels
phi = featurize(logits)

combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
s_all = combiner.score(phi)
print("combiner fit reproduced from notebook 07 (same seed, same split file, same features).")

combiner_fit n=1500  threshold_cal n=300  id_test n=1500


combiner fit reproduced from notebook 07 (same seed, same split file, same features).


In [3]:
def ece(scores, labels_, n_bins=10):
    bins = torch.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m_ = (scores >= lo) & (scores < hi)
        if m_.sum() == 0:
            continue
        conf = scores[m_].mean()
        acc = labels_[m_].float().mean()
        total += m_.float().mean() * (conf - acc).abs()
    return total.item()

# Baseline: exact reproduction of the DESIGN.md-reported number, as a sanity check
# that this notebook's pipeline matches notebook 07's before trusting anything new.
temp_cal_baseline = TemperatureScaling().fit(s_all[m_cal], correct[m_cal].float())
ece_before_baseline = ece(s_all[m_test], correct[m_test])
ece_after_baseline = ece(temp_cal_baseline.transform(s_all)[m_test], correct[m_test])
print(f"Baseline reproduction: ECE before = {ece_before_baseline:.4f}, after = {ece_after_baseline:.4f}")
print("(DESIGN.md \u00a715.2 reports 0.175 / 0.179 - this should match closely.)")

Baseline reproduction: ECE before = 0.1751, after = 0.1788
(DESIGN.md §15.2 reports 0.175 / 0.179 - this should match closely.)


## Bootstrap-resample `threshold_cal` at several sizes

For each candidate size, draw 30 independent bootstrap resamples (with replacement) of
`threshold_cal`'s own `(s, y)` pairs, fit temperature scaling on each, and measure ECE on
the same, never-resampled `id_test`. Report the mean and standard deviation across the 30
draws per size, not a single noisy draw.

In [4]:
rng = np.random.default_rng(0)
cal_idx = torch.nonzero(m_cal, as_tuple=True)[0]
s_cal_real, y_cal_real = s_all[cal_idx], correct[cal_idx].float()
s_test, y_test = s_all[m_test], correct[m_test]

sizes = [150, 300, 600, 1200, 2400]
n_draws = 30
results = {}

for size in sizes:
    eces = []
    for draw in range(n_draws):
        boot_idx = rng.integers(0, len(cal_idx), size=size)
        s_boot = s_cal_real[boot_idx]
        y_boot = y_cal_real[boot_idx]
        temp_cal = TemperatureScaling().fit(s_boot, y_boot)
        s_test_cal = temp_cal.transform(s_test)
        eces.append(ece(s_test_cal, y_test))
    eces = np.array(eces)
    results[size] = (eces.mean(), eces.std())
    print(f"size={size:5d}  ECE after scaling = {eces.mean():.4f} +/- {eces.std():.4f}  (n_draws={n_draws})")

print(f"\nFor reference: ECE with no calibration at all = {ece_before_baseline:.4f}")

size=  150  ECE after scaling = 0.1791 +/- 0.0036  (n_draws=30)


size=  300  ECE after scaling = 0.1787 +/- 0.0018  (n_draws=30)


size=  600  ECE after scaling = 0.1792 +/- 0.0019  (n_draws=30)


size= 1200  ECE after scaling = 0.1787 +/- 0.0009  (n_draws=30)


size= 2400  ECE after scaling = 0.1787 +/- 0.0006  (n_draws=30)

For reference: ECE with no calibration at all = 0.1751


In [5]:
# Explicit pass/fail-style check rather than eyeballing the printed numbers:
# does ECE-after-scaling trend meaningfully downward (toward ece_before, i.e. improving)
# as size grows from 150 to 2400 (16x), or does it stay flat?
means = np.array([results[s][0] for s in sizes])
biggest_vs_smallest = means[0] - means[-1]
print(f"ECE at smallest size (150): {means[0]:.4f}")
print(f"ECE at largest size (2400, 8x the real split): {means[-1]:.4f}")
print(f"Change: {biggest_vs_smallest:+.4f}")
if abs(biggest_vs_smallest) < 0.01:
    print("\nCONCLUSION: ECE stays essentially flat across an 8x range of calibration-fit")
    print("sample size. This is real evidence AGAINST the sample-size-artifact hypothesis")
    print("and FOR DESIGN.md's existing explanation (the backbone is already close to")
    print("well-calibrated, leaving little real miscalibration for more data to help fix).")
else:
    print("\nCONCLUSION: ECE changes meaningfully with calibration-fit sample size.")
    print("This is evidence that the original null result WAS at least partly a")
    print("sample-size artifact, not purely a 'nothing to calibrate' finding - the")
    print("existing DESIGN.md explanation needs qualifying, not just repeating.")

ECE at smallest size (150): 0.1791
ECE at largest size (2400, 8x the real split): 0.1787
Change: +0.0004

CONCLUSION: ECE stays essentially flat across an 8x range of calibration-fit
sample size. This is real evidence AGAINST the sample-size-artifact hypothesis
and FOR DESIGN.md's existing explanation (the backbone is already close to
well-calibrated, leaving little real miscalibration for more data to help fix).


## An honest limitation of this diagnostic

Bootstrap resampling with replacement, even at size 2400, is still only drawing repeatedly
from the same 300 real, unique `threshold_cal` points - it tests whether the *optimizer* was
starved of enough data to fit stably, not whether genuinely more/different real calibration
images would reveal miscalibration invisible to resampling the same 300. This is the same
class of pitfall `DESIGN.md` §23.2 already caught once for the conformal-bound validation
(bootstrap resampling breaks the i.i.d./independence assumption a real sample-size increase
would satisfy). The precise, honest conclusion: **this rules out simple fitting/optimizer
instability from few real points as the explanation for the null result**, and is consistent
with `DESIGN.md`'s existing "already well-calibrated" account - but it does not, and cannot,
rule out that genuinely more independent Imagenette images (beyond what this project has)
might reveal real miscalibration this diagnostic structurally cannot see.